# IIIF マニフェストファイルからの画像一括ダウンロード

IIIF マニフェストに記述された画像をまとめてダウンロードします。IIIF Presentation API **2.x と 3.0 の両方**に対応しており、Collection を指定すると含まれる Manifest を展開します。

> **ご利用にあたって**
> 画像の提供機関に配慮してご利用ください。マニフェストや画像に付された利用条件（`rights` / `attribution` / `requiredStatement` など）を確認のうえ、ご利用ください。
> 既定では 1 画像ごとに 1 秒待機します。大量に取得する場合は待機秒数を長めにしてください。

## 使い方

上のセルから順に実行してください。設定を変更するのは「2. 設定」のセルだけです。

1. インストール（そのまま実行）
2. 設定（マニフェストの URL などを変更）
3. ダウンロード（そのまま実行）
4. 確認（そのまま実行、任意）
5. 圧縮してローカルに保存（そのまま実行、任意）

## ソースコード

処理の本体は Python パッケージとして GitHub で管理しています。ノートブック外（ローカルやサーバ）でも同じ処理をコマンドラインから実行できます。

- パッケージ: https://github.com/nakamura196/000_tools/tree/main/packages/iiif-image-downloader
- 変更履歴: https://github.com/nakamura196/000_tools/blob/main/packages/iiif-image-downloader/CHANGELOG.md

## 1. インストール

In [ ]:
!pip install -q "git+https://github.com/nakamura196/000_tools.git#subdirectory=packages/iiif-image-downloader"

import iiif_image_downloader

print("iiif-image-downloader", iiif_image_downloader.__version__)

## 2. 設定

`manifests` に、ダウンロードしたいマニフェストの URL を指定してください。Collection の URL を指定すると、含まれる Manifest がすべて対象になります。

サンプルとして、国立国会図書館「校異源氏物語」（Presentation API 2.x）と IIIF Cookbook のサンプル（Presentation API 3.0）を挙げています。

In [ ]:
# ダウンロード対象。Manifest でも Collection でも指定できます。
manifests = [
    "https://www.dl.ndl.go.jp/api/iiif/3437686/manifest.json",
    "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json",
]

# 各マニフェストからダウンロードする画像の上限。すべて取得する場合は -1。
size_limit = 3

# 各リクエストの前に待機する秒数。提供機関のサーバに負荷をかけないための設定です。
sleep_seconds = 1.0

# 出力フォルダ名
output_dirname = "data"

# 画像サイズ（IIIF Image API の size）。
#   None … 自動（Image API 3 は "max"、1.x/2.x は "full"。原寸）
#   "!1024,1024" … 長辺 1024px 以内に収める
#   "1000," … 幅 1000px
image_size = None

# ファイル名に Canvas のラベルを付ける場合は True
use_label = False

# 既にあるファイルを再取得する場合は True（既定では存在すればスキップ）
overwrite = False

## 3. ダウンロード

取得できなかった画像があっても処理は止まらず、最後にまとめて表示します。中断した場合でも、同じセルをもう一度実行すれば未取得のものだけが取得されます。

In [ ]:
from iiif_image_downloader import DownloadOptions, download

options = DownloadOptions(
    output_dir=output_dirname,
    limit=size_limit,
    size=image_size,
    sleep=sleep_seconds,
    use_label=use_label,
    overwrite=overwrite,
)

report = download(manifests, options)

for manifest_report in report.manifests:
    print(
        "-",
        manifest_report.label or manifest_report.manifest_id,
        "(Presentation API {})".format(manifest_report.presentation_version),
        "->",
        manifest_report.output_dir,
    )
    if manifest_report.error:
        print("   ! ", manifest_report.error)
    for result in manifest_report.results:
        if result.status == "failed":
            print("   ! ", result.url, result.error)

print()
print(report.summary())

## 4. 確認（任意）

ダウンロードした画像の先頭数枚を表示します。

In [ ]:
import glob
import os

from IPython.display import Image, display

preview_count = 3

# ブラウザ上で表示できる形式のみプレビューします（TIFF などは件数のみ）。
displayable = (".jpg", ".jpeg", ".png", ".gif")
image_extensions = displayable + (".tif", ".tiff", ".jp2", ".webp", ".pdf")

paths = sorted(
    path
    for path in glob.glob(os.path.join(output_dirname, "**", "*"), recursive=True)
    if os.path.isfile(path) and path.lower().endswith(image_extensions)
)

shown = 0
for path in paths:
    if shown >= preview_count:
        break
    if not path.lower().endswith(displayable):
        continue
    print(path)
    display(Image(filename=path, width=320))
    shown += 1

print("{} file(s) in {}/".format(len(paths), output_dirname))

## 5. 圧縮してローカルに保存（任意）

Colab 上のファイルはセッションが終了すると消えます。手元に残す場合は次のセルを実行してください。

In [ ]:
import os
import shutil

if not os.path.isdir(output_dirname):
    print("{}/ がありません。先に「3. ダウンロード」を実行してください。".format(output_dirname))
else:
    # base_dir を指定して、zip の中に output_dirname のフォルダごと残します。
    zip_path = shutil.make_archive(output_dirname, "zip", ".", output_dirname)
    print("created:", zip_path)

    try:
        from google.colab import files

        files.download(zip_path)
    except ImportError:
        print("Colab 以外の環境では、上の zip ファイルを直接ご利用ください。")

## （参考）出力フォルダおよび zip ファイルの削除

やり直したいときに実行してください。

In [ ]:
import os
import shutil

shutil.rmtree(output_dirname, ignore_errors=True)
if os.path.exists(output_dirname + ".zip"):
    os.remove(output_dirname + ".zip")
print("removed:", output_dirname, "/", output_dirname + ".zip")

## （参考）コマンドラインからの利用

同じ処理は、インストール済みの環境で `iiif-image-download` コマンドとしても実行できます。取得予定の URL と保存先だけを確認する `--dry-run` などのオプションがあります。

In [ ]:
!iiif-image-download --help